# k8b — Ứng viên khó + nhãn teacher cho câu hỏi tổng hợp (GPU T4 ×2, vLLM)

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On.
**Add Input:** `mir2-code`, `mir2-data`, `mir2-chunks`, `mir2-synth` (output k8a), output k7 có `runs/synth_lin07.parquet`.
Mỗi câu tổng hợp: đoạn nguồn + đoạn tốt nhất của top-`TOP` tài liệu (âm khó) + `FAR` tài liệu ngẫu nhiên hạng 50–200
(âm dễ); bỏ bản gần trùng đoạn nguồn. Teacher chấm 0/1/2 theo thang `docs/EVAL_PLAN.md` §2.3 (tắt chế độ suy nghĩ), lấy
xác suất từng mức từ logprob → nhãn mềm.
**Nhiều tài khoản:** cùng Input, `SHARD = '0/3'`, `'1/3'`, `'2/3'` (ứng viên tính lại giống hệt ở mọi phiên).
**Chạy thử:** `LIMIT = 30` (30 câu) → xem cặp/s ở log để chia phần.
**Đầu ra:** `synth/pairs.parquet`, `synth/labels-<phần>.parquet` → *New Dataset* **`mir2-labels-<phần>`** → k8c.
**vLLM trên T4:** cài ở ô (2) (`VLLM_SPEC`, mặc định bản mới nhất). Lỗi khi nạp model → thử `VLLM_SPEC =
'vllm==0.9.2'` (hoặc bản cũ hơn có Qwen3), hoặc `MODEL = 'Qwen/Qwen3-14B-AWQ'` (nhanh ~2 lần, chất lượng thấp hơn).

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k8b-v1 (15/10/2026)'
MODEL = 'Qwen/Qwen3-32B-AWQ'
TOP, FAR = 12, 4
SHARD = '0/1'
LIMIT = 30                           # >0: chỉ N câu đầu của phần (đo tốc độ); 0 = cả phần
VLLM_SPEC = 'vllm'
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
def _ver(h):                                                  # output notebook cũ (vd mir2-chunks/code) cũng có mirret/
    v = os.path.join(os.path.dirname(os.path.dirname(h)), 'VERSION.txt')
    return open(v, encoding='utf-8').read().strip() if os.path.exists(v) else ''
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True),
              key=lambda h: (_ver(h), -len(h)), reverse=True)   # bản mã mới nhất theo VERSION.txt (ngày giờ)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=('chunks',))

In [ ]:
# (2) Dữ liệu k8a + run gộp của k7 → ứng viên (CPU)
for d in locate_all('synth/source.parquet'):
    shutil.copytree(f'{d}/synth', f'{K.work}/synth', dirs_exist_ok=True)
for d in locate_all('qexp/synth/queries_vi.parquet'):
    shutil.copytree(f'{d}/qexp/synth', f'{K.work}/qexp/synth', dirs_exist_ok=True)
RUN = locate('runs/synth_lin07.parquet') / 'runs' / 'synth_lin07.parquet'
print('run:', RUN)
sh(f'python -m mirret.synth candidates --config {K.cfg} --run {RUN} --top {TOP} --far {FAR}', 'candidates')

In [ ]:
# (3) Teacher chấm (2 GPU, tensor parallel)
sh(f'python -m pip install -q "{VLLM_SPEC}"', 'pip_vllm')
lim = f' --limit {LIMIT}' if LIMIT else ''
sh(f'python -m mirret.synth label --config {K.cfg} --model {MODEL} --tp 2 --shard {SHARD}{lim}', 'label')
report('k8b')